# Baseline Augmentation Sweep

This notebook contains 8 model-config pairs (2 models × 4 augmentations).
Each run cell executes the baseline evaluation (no EViT/ToMe pruning) for that single pair.

**Models:** `vit_tiny_patch16_224` (vit_t), `vit_base_patch16_224` (vit_b)

**Augmentations:** `horizontal_flip`, `color_jitter`, `rotation`, `gaussian_noise`

In [ ]:
# Optional for fresh Colab runtime
!git clone https://github.com/Chalhotra/ViT-Token-Economy.git
%cd ViT-Token-Economy

In [ ]:
!git checkout priyanshu/augmentation
!pip -q install -r requirements.txt
!pip -q install -e .

In [ ]:
from augmentation import horizontal_flip_only, jitter_only, rotation_only, gaussian_noise_only
from src.imagenet_mapping import build_imagenet100_to_1k_map
from src.models import ModelConfig, create_model, shrink_imagenet1k_head_to_imagenet100
from src.data import DataConfig, load_imagenet100_split, build_transform_for_model, apply_timm_preprocess, build_loader, make_collate_fn
from src.eval import evaluate_accuracy_latency_throughput, evaluate_with_topk_predictions, compute_gflops
from src.utils import get_device, num_params
from pathlib import Path
from datetime import datetime
import json
import pandas as pd
from PIL import Image

In [ ]:
device = get_device()
maps = build_imagenet100_to_1k_map()
print(f'Using device: {device}')

In [ ]:
AUGMENTATION_BUILDERS = {
    'horizontal_flip': horizontal_flip_only,
    'color_jitter': jitter_only,
    'rotation': rotation_only,
    'gaussian_noise': gaussian_noise_only,
}

if 'results' not in globals():
    results = []

In [ ]:
OUTPUT_DIR = Path.cwd() / 'lbp_outputs'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
RUNS_CSV = OUTPUT_DIR / 'baseline_runs.csv'
PREDICTIONS_CSV = OUTPUT_DIR / 'baseline_predictions.csv'

RUNS_COLUMNS = [
    'run_id', 'model_id', 'augmentation', 'aug_params',
    'acc1', 'gflops', 'latency_ms', 'params_m',
    'throughput', 'timestamp',
]
PREDICTIONS_COLUMNS = [
    'run_id', 'image_id', 'ground_truth_label', 'rank', 'predicted_class',
    'confidence', 'is_correct', 'entropy', 'conf_gap_to_rank1',
]

def _append_csv_rows(csv_path, rows, dedupe_subset):
    frame = pd.DataFrame(rows)
    if csv_path.exists():
        existing = pd.read_csv(csv_path)
        frame = pd.concat([existing, frame], ignore_index=True)
        frame = frame.drop_duplicates(subset=dedupe_subset, keep='last')
    frame.to_csv(csv_path, index=False)

def run_baseline_test(model_id, aug_builder=horizontal_flip_only, aug_label='horizontal_flip', batch_size=64, collect_predictions=False):
    print('\n' + '=' * 80)
    print(f'Testing baseline: {model_id}')
    print(f'Augmentation: {aug_label}')
    print('(No EViT / ToMe pruning applied)')
    print('=' * 80 + '\n')

    model = create_model(ModelConfig(model_id=model_id, pretrained=True))
    model = shrink_imagenet1k_head_to_imagenet100(model, maps.new_to_old_map, num_classes=100)
    model = model.to(device).eval()

    print("➡️ Loading dataset...")
    ds = load_imagenet100_split(DataConfig(split='validation'))
    class_names = ds.features['label'].names if hasattr(ds.features['label'], 'names') else [str(i) for i in range(100)]
    print("➡️ Building transform...")
    transform = build_transform_for_model(model)
    print("➡️ Building augmentation pipeline...")
    aug_pipeline = aug_builder() if callable(aug_builder) else aug_builder
    print("➡️ Applying preprocess (this may take time)...")
    ds_t, transform = apply_timm_preprocess(ds, transform, aug_pipeline=aug_pipeline)
    print("➡️ Creating collate_fn...")
    collate_fn = make_collate_fn(transform)
    print("➡️ Building loader...")
    loader = build_loader(ds_t, DataConfig(batch_size=batch_size, split='validation', shuffle=False), collate_fn=collate_fn)

    print("➡️ Starting evaluation...")
    if collect_predictions:
        print("Storing predictions also")
        metrics, prediction_rows = evaluate_with_topk_predictions(model, loader, device, class_names=class_names, topk=10)
    else:
        metrics = evaluate_accuracy_latency_throughput(model, loader, device)
        prediction_rows = []

    sample = transform(ds_t[0]["pixel_values"]).unsqueeze(0).to(device)

    print("➡️ Computing GFLOPs...")
    gflops = compute_gflops(model, sample)

    result = {
        'model': model_id,
        'augmentation': aug_label,
        'params_m': num_params(model) / 1e6,
        'gflops': gflops,
        **metrics,
    }

    print(f"Results for {model_id} | {aug_label}:")
    print(f"  Top-1 Accuracy : {metrics['acc1']:.2f}%")
    print(f"  GFLOPs         : {gflops:.3f}")
    print(f"  Latency        : {metrics['latency_ms']:.2f} ms")
    print(f"  Throughput     : {metrics['throughput']:.1f} samples/sec")
    return (result, prediction_rows) if collect_predictions else result

def run_single_baseline(model_id, aug_name, aug_builder, batch_size=64):
    run_key = f'baseline__{model_id}__{aug_name}'

    global results
    results = [x for x in results if x.get('run_id') != run_key]

    result, prediction_rows = run_baseline_test(
        model_id=model_id,
        aug_builder=aug_builder,
        aug_label=aug_name,
        batch_size=batch_size,
        collect_predictions=True,
    )

    run_id = run_key
    timestamp = datetime.utcnow().replace(microsecond=0).isoformat() + 'Z'
    aug_params = json.dumps({'augmentation': aug_name, 'batch_size': batch_size}, sort_keys=True)

    result['run_id'] = run_id
    result['model_id'] = model_id
    result['augmentation'] = aug_name
    result['aug_params'] = aug_params
    result['timestamp'] = timestamp

    run_record = {column: result.get(column) for column in RUNS_COLUMNS}
    _append_csv_rows(RUNS_CSV, [run_record], dedupe_subset=['run_id'])

    prediction_records = []
    for row in prediction_rows:
        row = dict(row)
        row['run_id'] = run_id
        prediction_records.append({column: row.get(column) for column in PREDICTIONS_COLUMNS})

    _append_csv_rows(PREDICTIONS_CSV, prediction_records, dedupe_subset=['run_id', 'image_id', 'rank'])

    print(f'Saved run summary to {RUNS_CSV}')
    print(f'Saved top-10 predictions to {PREDICTIONS_CSV}')

    results.append(result)
    print(f'Stored result for {model_id} | {aug_name}')

## Baseline Runs (8 Cells)
Run cells independently. Each run cell executes one tuple: (model, augmentation) with no token reduction.

In [ ]:
# vit_t | horizontal_flip
run_single_baseline(
    model_id='vit_tiny_patch16_224',
    aug_name='horizontal_flip',
    aug_builder=horizontal_flip_only,
)

In [ ]:
# vit_t | color_jitter
run_single_baseline(
    model_id='vit_tiny_patch16_224',
    aug_name='color_jitter',
    aug_builder=jitter_only,
)

In [ ]:
# vit_t | rotation
run_single_baseline(
    model_id='vit_tiny_patch16_224',
    aug_name='rotation',
    aug_builder=rotation_only,
)

In [ ]:
# vit_t | gaussian_noise
run_single_baseline(
    model_id='vit_tiny_patch16_224',
    aug_name='gaussian_noise',
    aug_builder=gaussian_noise_only,
)

In [ ]:
# vit_b | horizontal_flip
run_single_baseline(
    model_id='vit_base_patch16_224',
    aug_name='horizontal_flip',
    aug_builder=horizontal_flip_only,
)

In [ ]:
# vit_b | color_jitter
run_single_baseline(
    model_id='vit_base_patch16_224',
    aug_name='color_jitter',
    aug_builder=jitter_only,
)

In [ ]:
# vit_b | rotation
run_single_baseline(
    model_id='vit_base_patch16_224',
    aug_name='rotation',
    aug_builder=rotation_only,
)

In [ ]:
# vit_b | gaussian_noise
run_single_baseline(
    model_id='vit_base_patch16_224',
    aug_name='gaussian_noise',
    aug_builder=gaussian_noise_only,
)